# Twilight window extraction

In [1]:
import os
import numpy as np
from astropy import units as u
from astropy.io import fits
from astropy.table import Table
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import skyfield as sf
from skyfield.api import load, wgs84, EarthSatellite, N, W,S,E, Star, Angle
import datetime as dt
from skyfield import almanac
from skyfield.framelib import galactic_frame
from skyfield.data import hipparcos

from src.satellites.starlink_sat import *
from src.utils.calculations import *
from src.utils.plots import *
from src.utils.utils import *
from src.tracking.satellite_tracker import SatelliteTracker
from src.tracking.target_tracker import TargetTracker
from src.tracking.exposure_tracker import ExposureTracker

from astropy.wcs import WCS
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from matplotlib import colors as mcolors
from astropy.visualization import quantity_support, time_support
quantity_support()  

now = datetime.now()
cwd = os.getcwd()
print(cwd)

/data/a.saricaoglu/repo/RubinsForge/Skyfield/scripts


### Here, we determine the time period to extract twilight windows from.

In [10]:
ts = load.timescale()
year = 2026
month = 9
day_month_i = [1, month]
day_month_f = [30, month]
zone_stl = timezone('Etc/GMT-6')
zone = timezone('Chile/Continental')
utc = timezone('UTC')
obs_start = dt.datetime(year,day_month_i[1], day_month_i[0],0,0,0,0)
obs_end = dt.datetime(year,day_month_f[1], day_month_f[0],0,0,0,0)
local_start = ts.from_datetime(zone.localize(obs_start).astimezone(utc))
local_end = ts.from_datetime(zone.localize(obs_end).astimezone(utc))
#starlinks = load.tle_file('https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle')
eph = load('de421.bsp')
earth = eph['earth']
sun = eph['sun']
rubin_obs = wgs84.latlon(-30.244633,  -70.749417, elevation_m = 2647)
rubinobs_astr = earth +  wgs84.latlon(30.244633*S,  70.749417*W, elevation_m = 2647)

### Extracting windows using Skyfield funtions

In [11]:
today = local_start
next_day = local_end
f = sf.almanac.dark_twilight_day(eph, rubin_obs)
times, events = almanac.find_discrete(today, next_day, f)
previous_e = f(local_start).item()
twilight_events = []
for t, e in zip(times, events):
    tstr = str(t.astimezone(zone))[:16]
    mjd_time = Time(t.ut1, format='jd').mjd
    twilight=[]
    if previous_e < e:
        print(f'{tstr} mjd {mjd_time} {almanac.TWILIGHTS[e]} starts')
        if str(almanac.TWILIGHTS[e]) == 'Day':
            day_start = t # Twilight ends for morning
            twilight.append(('am twilight end', tstr,mjd_time))
        if str(almanac.TWILIGHTS[e]) == 'Astronomical twilight':
            night_end = t # Twilight starts for morning
            twilight.append(('am twilight start', tstr,mjd_time))
    else:
        print(f'{tstr} mjd {mjd_time} {almanac.TWILIGHTS[previous_e]} ends')
        if (almanac.TWILIGHTS[previous_e]) == 'Day':
            day_end = t # Twilight starts for evening
            twilight.append(('pm twilight start', tstr,mjd_time))
        if (almanac.TWILIGHTS[previous_e]) == 'Astronomical twilight':
            night_start = t # Twilight ends for evening
            twilight.append(('pm twilight end', tstr,mjd_time))
    if twilight:
        twilight_events.append(twilight)
    previous_e = e


2026-09-01 05:38 mjd 61284.401830455754 Astronomical twilight starts
2026-09-01 06:06 mjd 61284.42111318745 Nautical twilight starts
2026-09-01 06:34 mjd 61284.44044758938 Civil twilight starts
2026-09-01 06:58 mjd 61284.45719231386 Day starts
2026-09-01 18:27 mjd 61284.936070568394 Day ends
2026-09-01 18:52 mjd 61284.95282712346 Civil twilight ends
2026-09-01 19:19 mjd 61284.9721782743 Nautical twilight ends
2026-09-01 19:47 mjd 61284.99148197286 Astronomical twilight ends
2026-09-02 05:37 mjd 61285.40103574237 Astronomical twilight starts
2026-09-02 06:05 mjd 61285.42031675996 Nautical twilight starts
2026-09-02 06:33 mjd 61285.4396408638 Civil twilight starts
2026-09-02 06:57 mjd 61285.45636975672 Day starts
2026-09-02 18:28 mjd 61285.93644756358 Day ends
2026-09-02 18:52 mjd 61285.95318877138 Civil twilight ends
2026-09-02 19:20 mjd 61285.97253028769 Nautical twilight ends
2026-09-02 19:48 mjd 61285.99183308706 Astronomical twilight ends
2026-09-03 05:36 mjd 61286.400232608896 Astr

Cehcking if the window pairs are created correctly

In [12]:
print(twilight_events)
twilight_starts = [event for event in twilight_events if 'twilight start' in event[0][0]]
print("Twilight starts:", twilight_starts)
twilight_ends = [event for event in twilight_events if 'twilight end' in event[0][0]]
print("Twilight ends:", twilight_ends)
for start, end in zip(twilight_starts, twilight_ends):
    print(f"Twilight period from {start[0][1]} to {end[0][1]}")


[[('am twilight start', '2026-09-01 05:38', np.float64(61284.401830455754))], [('am twilight end', '2026-09-01 06:58', np.float64(61284.45719231386))], [('pm twilight start', '2026-09-01 18:27', np.float64(61284.936070568394))], [('pm twilight end', '2026-09-01 19:47', np.float64(61284.99148197286))], [('am twilight start', '2026-09-02 05:37', np.float64(61285.40103574237))], [('am twilight end', '2026-09-02 06:57', np.float64(61285.45636975672))], [('pm twilight start', '2026-09-02 18:28', np.float64(61285.93644756358))], [('pm twilight end', '2026-09-02 19:48', np.float64(61285.99183308706))], [('am twilight start', '2026-09-03 05:36', np.float64(61286.400232608896))], [('am twilight end', '2026-09-03 06:55', np.float64(61286.455541759264))], [('pm twilight start', '2026-09-03 18:29', np.float64(61286.93682404421))], [('pm twilight end', '2026-09-03 19:48', np.float64(61286.992186691146))], [('am twilight start', '2026-09-04 05:35', np.float64(61287.39942132309))], [('am twilight end

### Creating the npz file

In [13]:
import numpy as np

def save_twilight_windows_numpy(twilight_events, filename_base):
    """Save twilight windows to numpy format"""
    
    # Extract MJD times for efficient storage
    twilight_mjds = []
    twilight_types = []
    
    for event_group in twilight_events:
        for event in event_group:
            twilight_types.append(event[0])  # event type
            twilight_mjds.append(event[2])   # MJD time
    
    # Save as structured array
    twilight_array = np.array(list(zip(twilight_types, twilight_mjds)), 
                             dtype=[('type', 'U20'), ('mjd', 'f8')])
    
    np.savez_compressed(f"{filename_base}.npz", 
                       twilight_data=twilight_array,
                       metadata=np.array([local_start, local_end]))
    
    print(f"Twilight data saved to {filename_base}.npz")

def load_twilight_windows_numpy(filename):
    """Load twilight windows from numpy format"""
    data = np.load(filename)
    twilight_data = data['twilight_data']
    metadata = data['metadata']
    
    print(f"Loaded {len(twilight_data)} twilight events")
    print(f"Year: {int(metadata[0])}, Month: {int(metadata[1])}")
    
    return twilight_data, metadata

# Save the data
numpy_filename = f"twilight_windows_{local_start.astimezone(zone).strftime('%Y%m%dT%H%M')}_to_{local_end.astimezone(zone).strftime('%Y%m%dT%H%M')}"
save_twilight_windows_numpy(twilight_events, numpy_filename)

Twilight data saved to twilight_windows_20260901T0000_to_20260930T0000.npz
